### Sample 10% of videos from outputs folder

In [2]:
import os
import random
import math

In [ ]:


base_dir = '/Users/eveyhuang/Documents/NICO/gemini_code/outputs'

# Collect all JSON files and map them to their video file names
video_file_map = {}
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith('.json') and not file.startswith('all_') and not file.startswith('verbal_'):
            video_name = file.replace('.json', '')
            full_path = os.path.join(root, file)
            if video_name not in video_file_map:
                video_file_map[video_name] = []
            video_file_map[video_name].append(full_path)

# Get all unique video names
all_video_names = list(video_file_map.keys())
print(f"Total unique video names found: {len(all_video_names)}")
n_sample = max(1, math.ceil(0.1 * len(all_video_names)))  # At least 1

print(f"Number of videos to sample: {n_sample}")
# Randomly sample 10% of video names
random.seed(42)  # For reproducibility
sampled_video_names = random.sample(all_video_names, n_sample)

# Get all file paths for the sampled videos
sampled_file_paths = []
for name in sampled_video_names:
    sampled_file_paths.extend(video_file_map[name])

# Print or save the sampled file paths
print("Sampled JSON files for verification:")
for path in sampled_file_paths:
    print(path)

# Optionally, save to a text file
with open('sampled_json_files.txt', 'w') as f:
    for path in sampled_file_paths:
        f.write(path + '\n')

Total unique video names found: 781
Number of videos to sample: 79
Sampled JSON files for verification:
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MND/output_2021_04_22_MND_S6/Breakout_Room_4_Part_2_2021_04_22_13_14_53/Breakout_Room_4_Part_2_2021_04_22_13_14_53_chunk6.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021ABI/output_2021_05_21_ABI_S5/bot5p3_Room_5_Zoom_Meeting_5_21_2021_10_59_19_AM/bot5p3_Room_5_Zoom_Meeting_5_21_2021_10_59_19_AM_chunk3.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021ABI/output_2021_05_20_ABI_S4/bot2p2_Zoom_Meeting_2021_05_20_12_37_07/bot2p2_Zoom_Meeting_2021_05_20_12_37_07_chunk3.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.2_Zoom_Meeting_Room_1_2021_10_01_11_07_45/B1.2_Zoom_Meeting_Room_1_2021_10_01_11_07_45.json
/Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021SLU/output_2021_06_10_SLU_S5/botB2_2021_06_10_12_35_06/botB2_2021_06_10_12_35_06_chunk2.json
/Users/eveyh

In [9]:
import json
import os
import re
sampled_dict = {}

with open('sampled_json_files.txt', 'r') as f:
    file_paths = [line.strip() for line in f if line.strip()]

def extract_key_and_subkey(path):
    # Get the part after '/outputs/'
    rest = path.split('/outputs/')[1]
    parts = rest.split(os.sep)
    key = os.path.join(parts[0], parts[1])
    sub_key = os.path.join(*parts[2:])  # Join everything after the key
    return key, sub_key

def find_verbal_annotations(data, file):
    for dt in file:
        if dt["transcript"] == data["transcript"]:
            data["annotations"] = dt["annotations"]
    return data


for path in file_paths:
    # Extract key after 'output_'
    try:
        key, sub_key = extract_key_and_subkey(path)
        
    except Exception as e:
        print(f"Skipping {path}: {e}")
        continue
    
    folder, filename = os.path.split(path)
    filename = re.sub(r'_chunk\d+(?=\.json)', '', filename)
    # Load JSON and sample a value
    try:
        with open(path, 'r') as jf:
            data = json.load(jf)
        with open(os.path.join(folder, 'all_'+filename), 'r') as f:
            all_data = json.load(f)
        if isinstance(data, list) and data:
            sampled_value = random.choice(data)
        elif isinstance(data, dict) and data:
            filtered = [ann for ann in data["meeting_annotations"] if ann["speaking duration"] > 15]
            if filtered:
                sample_size = min(3, len(filtered))
                sampled_value = random.sample(filtered, sample_size)
                for sample in sampled_value:
                    sample = find_verbal_annotations(sample, all_data)
            else:
                # Fallback to any annotation if none meet the criteria
                sampled_value = None
        else:
            sampled_value = data
    except Exception as e:
        print(f"Error reading {path}: {e}")
        continue

    if key not in sampled_dict:
        sampled_dict[key] = {}
    sampled_dict[key][sub_key] = sampled_value

# Save to a new JSON file
with open('sampled_verification.json', 'w') as out_f:
    json.dump(sampled_dict, out_f, indent=2)

Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S3/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44_chunk3.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S3/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44/3_Sorbents_Zoom_Meeting_2020_11_05_12_20_44_chunk4.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_05_NES_S6/6_Theory_and_Expt_Zoom_Meeting_2020_11_05_10_28_39/6_Theory_and_Expt_Zoom_Meeting_2020_11_05_10_28_39_chunk1.json: 'speaking duration'
Error reading /Users/eveyhuang/Documents/NICO/gemini_code/outputs/2020NES/output_2020_11_06_NES_S7/1_beyond_co2

In [10]:
import pandas as pd

# Flatten sampled_verification for DataFrame
rows = []
for folder, files in sampled_dict.items():
    for file, samples in files.items():
        # samples can be a list or None
        if samples is None:
            rows.append({'folder': folder, 'file': file})
        elif isinstance(samples, list):
            for sample in samples:
                row = {'folder': folder, 'file': file}
                if isinstance(sample, dict):
                    row.update(sample)
                rows.append(row)
        elif isinstance(samples, dict):
            row = {'folder': folder, 'file': file}
            row.update(samples)
            rows.append(row)
        else:
            row = {'folder': folder, 'file': file, 'value': samples}
            rows.append(row)

df = pd.DataFrame(rows)
df.to_excel('sampled_verification.xlsx', index=False)

# Importing Evey's request

In [2]:
from pathlib import Path
BASE_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/outputs").expanduser()
RANDOM_SEED = 3839
OUT_CSV = BASE_DIR.parent / "sampling" / "sample_balanced.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [3]:
def find_all_gm_v4_files(base_dir: Path):
    return sorted(base_dir.rglob("all_gm_v4*.json"))

files = find_all_gm_v4_files(BASE_DIR)
print("Files found:", len(files))
assert files, "No all_gm_v4*.json files found under BASE_DIR; double-check the path."

Files found: 213


In [4]:
import json, pandas as pd
from typing import Any, Dict, List

def load_json_records(fp: Path) -> List[Dict[str, Any]]:
    try:
        with fp.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for v in data.values():
                if isinstance(v, list):
                    return v
    except json.JSONDecodeError:
        pass
    # NDJSON fallback
    recs = []
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    recs.append(obj)
            except json.JSONDecodeError:
                continue
    return recs

def extract_codes_from_item(item: dict) -> List[str]:
    # If labels/codes list exists, use it
    for key in ("labels", "codes"):
        if key in item and isinstance(item[key], list):
            return [str(x) for x in item[key] if isinstance(x, (str, int, float))]
    # Otherwise pull codes from annotations where score>0 (or value>0)
    ann = item.get("annotations")
    codes = []
    if isinstance(ann, dict):
        for k, v in ann.items():
            if isinstance(v, (bool, int, float)):
                if bool(v): codes.append(str(k))
            elif isinstance(v, dict):
                if "score" in v:
                    try:
                        if float(v["score"]) > 0: codes.append(str(k))
                    except Exception:
                        if bool(v["score"]): codes.append(str(k))
                elif "value" in v:
                    try:
                        if float(v["value"]) > 0: codes.append(str(k))
                    except Exception:
                        if bool(v["value"]): codes.append(str(k))
                elif any(bool(x) for x in v.values()):
                    codes.append(str(k))
    return codes

def infer_conference_from_path(p: Path, anchor="outputs") -> str:
    parts = list(p.parts)
    return parts[parts.index(anchor)+1] if anchor in parts and parts.index(anchor)+1 < len(parts) else "unknown"

def build_dataframe(file_paths: List[Path]) -> pd.DataFrame:
    rows = []
    for fp in file_paths:
        for i, rec in enumerate(load_json_records(fp)):
            rows.append({
                "conference": infer_conference_from_path(fp),
                "json_file": str(fp),
                "record_idx": i,
                "codes": extract_codes_from_item(rec),
                "speaker": rec.get("speaker"),
                "timestamp": rec.get("timestamp"),
                "transcript": rec.get("transcript"),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=["conference","json_file","record_idx","codes","speaker","timestamp","transcript"])
    return df

df = build_dataframe(files)
df["n_codes"] = df["codes"].apply(lambda x: len(x) if isinstance(x, list) else 0)
print("Utterances loaded:", len(df), "| conferences:", sorted(df["conference"].unique()))
print("Rows with empty codes:", int((df["n_codes"] == 0).sum()))

Utterances loaded: 30558 | conferences: ['2020NES', '2021ABI', '2021CMC', '2021MND', '2021MZT', '2021NES', '2021SLU', '2022MND']
Rows with empty codes: 4818


In [5]:
# wasn't sure why some of the annotations were called "None"...
def has_none_assigned(codes_list):
    return isinstance(codes_list, list) and ("None" in codes_list)
print("Rows with 'None' assigned (score>0):", int(df["codes"].apply(has_none_assigned).sum()))

Rows with 'None' assigned (score>0): 0


In [6]:
from collections import Counter
available_by_code = Counter(c for lst in df["codes"] for c in lst)
print("Availability per code:", dict(sorted(available_by_code.items())))

SAMPLES_PER_CODE = 100   # set 50–150 depending on desired sample size
print("Using SAMPLES_PER_CODE =", SAMPLES_PER_CODE)

Availability per code: {'Coordination and Decision Practices': 5199, 'Evaluation Practices': 3246, 'Idea Management': 6526, 'Information Seeking': 5659, 'Integration Practices': 1327, 'Knowledge Sharing': 11772, 'Participation Dynamics': 3120, 'Relational Climate': 6389}
Using SAMPLES_PER_CODE = 100


In [7]:
import random
from collections import Counter

def balanced_sample_strict(df, samples_per_code: int, seed: int):
    rng = random.Random(seed)
    work = df.copy()
    work["codes"] = work["codes"].apply(lambda x: x if isinstance(x, list) else [])
    all_codes = sorted({c for lst in work["codes"] for c in lst})
    if not all_codes:
        return work.sample(n=min(samples_per_code, len(work)), random_state=seed)

    codes_by_row = {i: set(lst) for i, lst in work["codes"].items()}
    idxs_by_code = {c: work.index[work["codes"].apply(lambda lst: c in lst)].tolist()
                    for c in all_codes}
    for c in all_codes:
        rng.shuffle(idxs_by_code[c])

    need = {c: samples_per_code for c in all_codes}
    selected, pool = set(), set().union(*idxs_by_code.values())

    def row_score(i):
        # +2 for underrepresented codes, -1 if already satisfied
        return sum(2 if need.get(c, 0) > 0 else -1 for c in codes_by_row[i])

    while any(n > 0 for n in need.values()) and pool:
        for c in sorted(need, key=lambda k: need[k], reverse=True):
            if need[c] <= 0: 
                continue
            candidates = [i for i in idxs_by_code[c] if i in pool and i not in selected]
            if not candidates:
                continue
            candidates.sort(key=lambda i: (row_score(i), -len(codes_by_row[i])), reverse=True)
            pick = candidates[0]
            selected.add(pick); pool.remove(pick)
            for cc in codes_by_row[pick]:
                if need.get(cc, 0) > 0:
                    need[cc] -= 1

        # stop if remaining rows can’t help any needed code
        if not any(i in pool and any(need.get(cc, 0) > 0 for cc in codes_by_row[i]) for i in pool):
            break

    sampled = work.loc[sorted(selected)].copy().reset_index(drop=True)
    per_code = Counter(c for lst in sampled["codes"] for c in lst)
    print("Target per code:", samples_per_code)
    print("Final per-code counts:", dict(sorted(per_code.items())))
    unmet = {c: n for c, n in need.items() if n > 0}
    if unmet:
        print("Could not reach target for some codes (too rare):", unmet)
    return sampled

In [8]:
sampled = balanced_sample_strict(df, SAMPLES_PER_CODE, RANDOM_SEED)
print("Rows in 'sampled':", len(sampled))

per_code = Counter(c for lst in sampled["codes"] for c in lst)
print("Per-code counts in FINAL SAMPLE:")
for k in sorted(per_code):
    print(f"{k}: {per_code[k]}")

# guardrail — ensures you didn’t accidentally save the full df
assert len(sampled) < len(df), "Oops—this looks like the full dataset. Lower SAMPLES_PER_CODE."

sampled.to_csv(OUT_CSV, index=False)
print("Wrote balanced sample to:", OUT_CSV)

Target per code: 100
Final per-code counts: {'Coordination and Decision Practices': 100, 'Evaluation Practices': 100, 'Idea Management': 100, 'Information Seeking': 100, 'Integration Practices': 100, 'Knowledge Sharing': 100, 'Participation Dynamics': 100, 'Relational Climate': 100}
Rows in 'sampled': 281
Per-code counts in FINAL SAMPLE:
Coordination and Decision Practices: 100
Evaluation Practices: 100
Idea Management: 100
Information Seeking: 100
Integration Practices: 100
Knowledge Sharing: 100
Participation Dynamics: 100
Relational Climate: 100
Wrote balanced sample to: /Users/maxchalekson/Desktop/gemini_data_analysis/sampling/sample_balanced.csv


In [9]:
print("\nConferences covered:", sorted(df["conference"].unique()))
print("\nPer-conference row counts (FINAL SAMPLE):")
print(sampled["conference"].value_counts().sort_index())


Conferences covered: ['2020NES', '2021ABI', '2021CMC', '2021MND', '2021MZT', '2021NES', '2021SLU', '2022MND']

Per-conference row counts (FINAL SAMPLE):
conference
2020NES    21
2021ABI    46
2021CMC    45
2021MND    43
2021MZT    26
2021NES    44
2021SLU    33
2022MND    23
Name: count, dtype: int64


## quick check - files dropped

In [13]:
# Find candidate "empty" files (where build_dataframe returned 0 utterances)
empty_files = [fp for fp in files if len(load_json_records(fp)) == 0]
print("Empty files detected:", len(empty_files))

# If any exist, inspect the first one
if empty_files:
    test_fp = empty_files[0]
    print("Inspecting file:", test_fp)

    try:
        with open(test_fp, "r", encoding="utf-8") as f:
            raw = f.read(1000)  # preview first 1000 chars
        print("\nFirst 1000 chars of file:\n", raw)
    except Exception as e:
        print("Error reading file:", e)
else:
    print("No empty files found.")

Empty files detected: 1
Inspecting file: /Users/maxchalekson/Desktop/gemini_data_analysis/outputs/2021MZT/output_2021_10_01_MZT_S1/B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18/all_gm_v4_B1.1_Zoom_Meeting_Room_1_2021_10_01_11_04_18.json

First 1000 chars of file:
 []
